In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Getting Started: Quick Gen AI Evaluation


 <table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fevaluation%2Fquick_start_gen_ai_eval.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/evaluation/quick_start_gen_ai_eval.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>


| Author(s) |
| --- |
| [Jason Dai](https://github.com/jsondai) |

## Overview

This notebook shows the quickest way to evaluate a single generative model using the Vertex AI SDK for Gen AI Eval Service.


### Costs

This tutorial uses billable components of Google Cloud:

- Vertex AI

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing) and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.


## Getting Started



In [2]:
#Due to Jupyter issue, isolate rendering function from sdk
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import json
import html
from vertexai._genai import types
import vertexai._genai._evals_visualization as sdk
from typing import Optional
from pydantic import errors
def display_evaluation_dataset(eval_dataset_obj: types.EvaluationDataset) -> None:
    """Displays an evaluation dataset in an IPython environment."""
    from IPython import display

    processed_rows = []
    df = eval_dataset_obj.eval_dataset_df

    for _, row in df.iterrows():
        processed_row = {}
        for col_name, cell_value in row.items():
            if col_name in ["prompt", "request", "response"]:
                processed_row[col_name] = sdk._extract_text_and_raw_json(cell_value)
            elif col_name == "rubric_groups":
                # Special handling for rubric_groups to keep it as a dict
                if isinstance(cell_value, dict):
                    processed_row[col_name] = {
                        k: [
                            (
                                v_item.model_dump(mode="json")
                                if hasattr(v_item, "model_dump")
                                else v_item
                            )
                            for v_item in v
                        ]
                        for k, v in cell_value.items()
                    }
                else:
                    processed_row[col_name] = cell_value
            else:
                if isinstance(cell_value, (dict, list)):
                    processed_row[col_name] = json.dumps(
                        cell_value, ensure_ascii=False, default=sdk._pydantic_serializer
                    )
                else:
                    processed_row[col_name] = cell_value
        processed_rows.append(processed_row)

    dataframe_json_string = json.dumps(processed_rows, ensure_ascii=False, default=str)
    html_content = sdk._get_inference_html(dataframe_json_string)
    #print(html_content)
    escaped_html = html.escape(html_content)
    iframe_code = f'<div><iframe srcdoc="{escaped_html}" width="100%" height="600px" style="border:none;"></iframe></div>'
    display.display(display.HTML(iframe_code))

def display_evaluation_result(eval_result_obj: types.EvaluationResult, candidate_names: Optional[list[str]] = None) -> None:
    """Displays evaluation result in an IPython environment."""
    from IPython import display

    try:
        result_dump = eval_result_obj.model_dump(
            mode="json", exclude_none=True, exclude={"evaluation_dataset"}
        )
    except errors.PydanticSerializationError as e:
        print(
            "Serialization Error: %s\nCould not display the evaluation "
            "result due to a data serialization issue. Please check the "
            "content of the EvaluationResult object.",
            e,
        )
        return
    except Exception as e:
        print("Failed to serialize EvaluationResult: %s", e, exc_info=True)
        raise

    input_dataset_list = eval_result_obj.evaluation_dataset
    is_comparison = input_dataset_list and len(input_dataset_list) > 1

    metadata_payload = result_dump.get("metadata", {})
    metadata_payload["candidate_names"] = candidate_names or metadata_payload.get(
        "candidate_names"
    )

    if is_comparison and input_dataset_list:
        if input_dataset_list[0]:
            metadata_payload["dataset"] = sdk._extract_dataset_rows(input_dataset_list[0])

        if "eval_case_results" in result_dump:
            for case_res in result_dump["eval_case_results"]:
                for resp_idx, cand_res in enumerate(
                    case_res.get("response_candidate_results", [])
                ):
                    if (
                        input_dataset_list is not None
                        and resp_idx < len(input_dataset_list)
                        and input_dataset_list[resp_idx]
                    ):
                        rows = sdk._extract_dataset_rows(input_dataset_list[resp_idx])
                        case_idx = case_res.get("eval_case_index")
                        if case_idx is not None and case_idx < len(rows):
                            original_case = rows[case_idx]
                            cand_res["display_text"] = original_case[
                                "response_display_text"
                            ]
                            cand_res["raw_json"] = original_case["response_raw_json"]

        win_rates = eval_result_obj.win_rates if eval_result_obj.win_rates else {}
        if "summary_metrics" in result_dump:
            for summary in result_dump["summary_metrics"]:
                if summary.get("metric_name") in win_rates:
                    summary.update(win_rates[summary["metric_name"]])

        result_dump["metadata"] = metadata_payload
        html_content = sdk._get_comparison_html(json.dumps(result_dump))
    else:
        single_dataset = input_dataset_list[0] if input_dataset_list else None
        processed_rows = []
        if single_dataset is not None:
            processed_rows = sdk._extract_dataset_rows(single_dataset)
            metadata_payload["dataset"] = processed_rows

            if "eval_case_results" in result_dump and processed_rows:
                for case_res in result_dump["eval_case_results"]:
                    case_idx = case_res.get("eval_case_index")
                    if (
                        case_idx is not None
                        and case_idx < len(processed_rows)
                        and case_res.get("response_candidate_results")
                    ):
                        original_case = processed_rows[case_idx]
                        cand_res = case_res["response_candidate_results"][0]
                        cand_res["display_text"] = original_case[
                            "response_display_text"
                        ]
                        cand_res["raw_json"] = original_case["response_raw_json"]

        result_dump["metadata"] = metadata_payload
        html_content = sdk._get_evaluation_html(json.dumps(result_dump))

    escaped_html = html.escape(html_content)
    iframe_code = f'<div><iframe srcdoc="{escaped_html}" width="100%" height="600px" style="border:none;"></iframe></div>'
    display.display(display.HTML(iframe_code))

In [3]:
# @title ### Set Google Cloud project information
# @markdown To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).
# @markdown  Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

import os
PROJECT_ID = "sandbox-373102"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
LOCATION= "us-central1"  # @param {type: "string", placeholder: "us-central1", isTemplate: true}
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", LOCATION)


from vertexai import Client, types
client = Client(project=PROJECT_ID, location=LOCATION)

In [4]:
# @title ### Generate Responses
# @markdown The eval workflow starts with `run_inference()` to generate model responses for your dataset. The SDK can automatically detects and handles several common data formats. This means you can often use your data as-is without needing to perform manual conversions


import pandas as pd

eval_df = pd.DataFrame({
    "prompt": [
        "Explain software 'technical debt' using a concise analogy of planting a garden.",
        "Write a Python function to find the nth Fibonacci number using recursion with memoization, but without using any imports.",
        "Write a four-line poem about a lonely robot, where every line must be a question and the word 'and' cannot be used.",
        "A drawer has 10 red socks and 10 blue socks. In complete darkness, what is the minimum number of socks you must pull out to guarantee you have a matching pair?",
        "An AI discovers a cure for a major disease, but the cure is based on private data it analyzed without consent. Should the cure be released? Justify your answer."
    ]
})

from google.genai import types as genai_types
from vertexai._genai import types
httpOptions = genai_types.HttpOptions(
    retry_options=genai_types.HttpRetryOptions(
        attempts=5,           # 최대 재시도 횟수
        initial_delay=1.0,    # 첫 대기 시간
        http_status_codes=[429, 500, 502, 503, 504] # 재시도 대상 에러 코드
    )
)

eval_dataset = client.evals.run_inference(
    model="gemini-2.5-flash",
    src=eval_df,
    config=types.EvalRunInferenceConfig(
        generate_content_config=genai_types.GenerateContentConfig(
            http_options=httpOptions
        )
    )
)

display_evaluation_dataset(eval_dataset)

Gemini Inference: 100%|██████████| 5/5 [00:23<00:00,  4.60s/it]


In [5]:
#https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/rubric-metric-details

data_with_rubrics = client.evals.generate_rubrics(
    src=eval_dataset,
    rubric_group_name="text_rubrics",
    predefined_spec_name=types.RubricMetric.TEXT_QUALITY,
    config=types.RubricGenerationConfig(http_options=httpOptions)
)

display_evaluation_dataset(data_with_rubrics)

In [6]:
# @title ### Run Evaluation
# @markdown Evaluate the responses using the GENERAL_QUALITY adaptive rubric-based metric by default.

eval_result = client.evals.evaluate(dataset=eval_dataset,
                                    metrics=[
                                        types.RubricMetric.TEXT_QUALITY,
                                        types.RubricMetric.FINAL_RESPONSE_QUALITY,
                                        types.RubricMetric.TOOL_USE_QUALITY,
                                        types.RubricMetric.HALLUCINATION,
                                        types.RubricMetric.SAFETY,
                                    ],
                                    config=types.EvaluateMethodConfig(
                                        http_options=httpOptions)
                                    )
display_evaluation_result(eval_result)

Computing Metrics for Evaluation Dataset:   0%|          | 0/25 [00:00<?, ?it/s]Metric 'tool_use_quality_v1' requires tool usage in 'intermediate_events' or 'agent_data', but no tool usage was found for case eval_case_0.
Metric 'tool_use_quality_v1' requires tool usage in 'intermediate_events' or 'agent_data', but no tool usage was found for case eval_case_1.
Metric 'tool_use_quality_v1' requires tool usage in 'intermediate_events' or 'agent_data', but no tool usage was found for case eval_case_2.
Metric 'tool_use_quality_v1' requires tool usage in 'intermediate_events' or 'agent_data', but no tool usage was found for case eval_case_3.
Metric 'tool_use_quality_v1' requires tool usage in 'intermediate_events' or 'agent_data', but no tool usage was found for case eval_case_4.
Computing Metrics for Evaluation Dataset: 100%|██████████| 25/25 [00:43<00:00,  1.75s/it]
